## Analysis of car market prices in India.
For this project, we chose the dataset available at: https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho?resource=download&select=Car+details+v3.csv

The dataset contains information about used cars listed for sale on Cardekho.com. It consists of 8128 rows and 13 columns. It is frequently used for price prediction tasks, making it a good example of applying linear regression in machine learning.

The main objective of the task is to examine the impact of various factors (number of owners, engine capacity, mileage, etc.) on the final selling price of cars in India.  

The dataset contains the following variables:

    name: Car name and model

    year: Year the car was purchased

    selling_price: Selling price of the car (target variable - this variable will act as the dependent variable)

    km_driven: Number of kilometers driven

    fuel: Fuel type (diesel, petrol, CNG, LPG)

    seller_type: Seller type (individual, dealer)

    transmission: Transmission type (manual, automatic)

    owner: Number of previous owners

    mileage: Mileage (fuel efficiency)

    engine: Engine capacity (CC)

    max_power: Engine max power (BHP)

    torque: Torque

    seats: Number of seats

Numerical data: Columns such as 'year', 'selling_price', 'km_driven', and 'seats' are in numerical format.

Text data: Columns 'name', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', and 'torque' are in text format and require conversion to numerical format to perform statistical analysis.

Missing values: The dataset contains missing values in the 'mileage', 'engine', 'max_power', and 'seats' columns, which account for about 3% of the data.

## Dataset Preparation

In [ ]:
%pip install --quiet matplotlib pandas numpy seaborn scikit-learn scipy
# If using this file in Google Colab, you should delete this cell or not execute it, I am doing this in vscode

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import datetime
from matplotlib.ticker import FuncFormatter
import scipy

df = pd.read_csv('Car details v3.csv', delimiter=',')
CURRENT_YEAR = datetime.date.today().year
print(df.head(5))

Data cleaning - removing NaN values:

In [ ]:
# checking data shape
print(f"Data shape before cleaning: {df.shape}")
# checking for NaNs
print(df.isnull().sum())
init_rows = len(df)

# cleaning dataset columns
df.dropna(subset=['selling_price', 'mileage', 'engine', 'max_power', 'torque', 'seats'], inplace=True)
df.reset_index(drop=True, inplace=True)
modded_rows = len(df)

print(f"\nRemoved {init_rows - modded_rows} rows due to NaNs in key columns.")
print(f"\nData shape after cleaning: {df.shape}")

Conversion of specific units into numerical variables:

In [ ]:
# Mileage: Removing units and converting to float
df['mileage'] = df['mileage'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)

# Engine: Removing "CC" and converting to float
df['engine'] = df['engine'].astype(str).str.extract(r'(\d+)').astype(float)

# Max Power: Removing "bhp", handling empty strings/issues and converting to float.
df['max_power'] = df['max_power'].astype(str).str.replace(r'\s*bhp', '', regex=True)
df['max_power'] = pd.to_numeric(df['max_power'], errors='coerce') # 'coerce' will turn non-numbers into NaN

In [ ]:
# Torque: Extracting numerical value and standardizing units
def clean_torque(torque_str):
    if pd.isna(torque_str):
        return np.nan
    torque_str = str(torque_str).lower().strip()
    val = np.nan

    # Attempting to extract value and unit - regular expressions
    match_nm_rpm = re.search(r'(\d+\.?\d*)\s*nm\s*@\s*([\d,]+-?[\d,]*)\s*rpm', torque_str)
    match_kgm_rpm = re.search(r'(\d+\.?\d*)\s*kgm\s*@\s*([\d,]+-?[\d,]*)\s*rpm', torque_str)
    match_nm_only = re.search(r'(\d+\.?\d*)\s*nm', torque_str) # For cases without RPM
    match_kgm_only = re.search(r'(\d+\.?\d*)\s*kgm', torque_str) # For cases without RPM
    match_num_first = re.search(r'^(\d+\.?\d*)', torque_str) # Last resort, if only a number is at the beginning

    if match_nm_rpm:
        val = float(match_nm_rpm.group(1))
    elif match_kgm_rpm:
        val = float(match_kgm_rpm.group(1)) * 9.80665
    elif match_nm_only:
        val = float(match_nm_only.group(1))
    elif match_kgm_only:
        val = float(match_kgm_only.group(1)) * 9.80665
    elif match_num_first: 
        # If it's just a number, we assume Nm, but this might be incorrect
        # Let's check if it is "number@number(kgm@rpm)" - unusual format
        weird_kgm_match = re.search(r'(\d+\.?\d*)\s*@\s*[\d,-]+\s*\(kgm@\s*rpm\)', torque_str)
        if weird_kgm_match:
            val = float(weird_kgm_match.group(1)) * 9.80665
        else:
            val = float(match_num_first.group(1)) # Final assignment
    return val

df['torque'] = df['torque'].apply(clean_torque)

In [ ]:
# --- Step 4: Processing the `name` column ---
df['brand'] = df['name'].apply(lambda x: x.split(' ')[0])
print("\n--- Most frequent brands (top 10) and 'Other' ---")
top_brands = df['brand'].value_counts().nlargest(10).index
df['brand'] = df['brand'].apply(lambda x: x if x in top_brands else 'Other')
print(df['brand'].value_counts())


In [ ]:
# --- Step 5: Processing the `owner` column ---
owner_map = {
    'First Owner': 1,
    'Second Owner': 2,
    'Third Owner': 3,
    'Fourth & Above Owner': 4,
    'Test Drive Car': 0
}
df['owner'] = df['owner'].map(owner_map)


One-Hot Encoding of the remaining categorical variables:

In [ ]:
# --- Step 6: Encoding remaining categorical variables ---
categorical_cols_to_encode = ['fuel', 'seller_type', 'transmission', 'brand']
df = pd.get_dummies(df, columns=categorical_cols_to_encode, drop_first=True, prefix=categorical_cols_to_encode)

In [ ]:
# --- Step 7: Creating a new feature `car_age` ---
df['car_age'] = CURRENT_YEAR - df['year']

In [ ]:
# --- Step 8: Dropping unnecessary or already processed columns ---
columns_to_drop_final = ['name', 'year']
df.drop(columns=columns_to_drop_final, inplace=True)

In [ ]:
# --- Step 9: Final check and NaN handling after transformations ---
print("\n--- Number of missing values AFTER transformations and BEFORE final imputation ---")
print(df.isnull().sum())

Filling NaNs with intermediate values (median):

In [ ]:
cols_to_impute_median_final = ['mileage', 'engine', 'max_power', 'torque', 'seats', 'owner']

for col in cols_to_impute_median_final:
    if col in df.columns and df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Imputed NaN in column '{col}' with median: {median_val}")

# Converting 'seats' to int, if possible and there are no NaNs left
if 'seats' in df.columns and df['seats'].isnull().sum() == 0:
    df['seats'] = df['seats'].astype(int)

In [ ]:
print("\n--- Number of missing values AFTER FINAL imputation ---")
print(df.isnull().sum())

print("\n--- Dataset info after cleaning ---")
df.info()
print("\n--- First 5 rows of the cleaned dataset ---")
pd.set_option('display.max_columns', None)
print(df.head())
print("\n--- Descriptive statistics of the cleaned dataset ---")
print(df.describe())

if df.isnull().all().any():
    print("\nWARNING: There are columns containing only NaN values!")
    print(df.isnull().all()[df.isnull().all()])
else:
    print("\nNo columns containing only NaNs.")

print("\n--- Final data shape: ---", df.shape)
cleaned_df = df
print("\nCleaning completed. Data is ready for further analysis.")

In summary:
1. Rows with missing values in key columns were removed.
2. Columns `mileage`, `engine`, `max_power`, and `torque` were standardized and converted to numeric types.
3. One-Hot Encoding was applied to categorical variables (`fuel`, `seller_type`, `transmission`, `brand`) so they could be used in the model.
4. A new feature `car_age` was created for better interpretation of age's impact on price.
5. The few remaining missing values were filled with the median to avoid losing observations.


# Descriptive statistics of the dataset:

In [ ]:
pd.set_option('display.float_format', lambda x: '%.5f' % x)
print("\n--- Dataset info after cleaning ---")
print(cleaned_df.info())
print("\n--- Descriptive statistics of the cleaned dataset ---")
print(f"{cleaned_df['selling_price'].describe(include='all').to_string()}\n")


After cleaning the dataset, we obtain 7906 rows containing information about the sales of used cars in India. The cheapest car costs 29,999 INR (Indian rupees), and the most expensive is 10,000,000 INR. The average price is approximately 649,813 INR, the median is 450,000 INR, and the first and third quartiles are 270,000 and 690,000 INR, respectively. The standard deviation is greater than the mean and amounts to approximately 813,582 INR.

# Charts / Visualizations 


In [ ]:
formatter = FuncFormatter(lambda x, pos: f'{int(x):,}')
plt.figure(figsize=(12, 6))
plt.title('Box plot of car sales (Selling Price)', fontsize=16)
ax = sns.boxplot(x=cleaned_df['selling_price'],native_scale=True,color='green', fliersize=5, linewidth=1.5)

# ax.set_xscale('log')  # Setting logarithmic scale for better visualization

ax.xaxis.set_major_formatter(formatter)
plt.xticks(rotation=45)
plt.xlabel('Selling Price (In INR)')
plt.show()

The box plot shows that most of the data is clustered on the left side, at lower prices. This means that the majority of the offered cars have a relatively low selling price, not exceeding 1,000,000 INR. The numerous outliers - the long "whisker" and numerous points on the right side - represent cars whose price is significantly higher than the average value. These are outliers. In a dataset based on vehicle sales, this is a completely natural phenomenon - the majority are popular, cheaper models, but the dataset also contains a small number of expensive, luxury cars that inflate the average and create such a "tail" on the chart. The box is relatively narrow and located at the beginning of the scale, confirming that 50% of all cars in the dataset fall within a fairly small and low price range.

In [ ]:
formatter = FuncFormatter(lambda x, pos: f'{int(x):,}')
plt.figure(figsize=(12, 6))
plt.title('Histogram - Selling Price')
ax = sns.histplot(x=cleaned_df['selling_price'],bins=50, kde=True,color='blue', edgecolor='black')

# ax.set_xscale('log')  # Setting logarithmic scale for better visualization

ax.xaxis.set_major_formatter(formatter)
plt.xticks(rotation=45)
plt.xlabel('Selling Price (In INR)')
plt.ylabel('Frequency')

plt.show()

The histogram confirms that the vast majority of cars are priced in the low range, probably below 1,000,000 INR. As the price increases, the number of cars drops sharply. One can also spot the presence of higher-end brands, whose price oscillates between 2,000,000 and 6,000,000 INR. Just like the box plot, the histogram clearly shows that the distribution is strongly positively skewed. In the context of prices, this is completely natural - most products on the market are affordable, and luxury products are rare.
There is a long "tail" on the right side, meaning there are very few cars with very high prices. Just like the box plot, the histogram clearly shows that the distribution is strongly positively skewed. This means that most cars in this market have an affordable price, and luxury products are rare, which is consistent with reality.

In [ ]:
plt.figure(figsize=(14, 7))

# Price depending on transmission type
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
sns.boxplot(x='transmission_Manual', y='selling_price', data=cleaned_df,color='orange')
plt.title('Selling Price vs. Transmission Type')
plt.xlabel('Is transmission manual? (1=Yes, 0=No)')
plt.ylabel('Selling Price (INR)')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(x):,}')) # Y-axis formatting

# Price depending on seller type
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
sns.boxplot(x='seller_type_Individual', y='selling_price', data=cleaned_df,color='purple')
plt.title('Selling Price vs. Seller Type')
plt.xlabel('Is seller an individual? (1=Yes, 0=No)')
plt.ylabel('') # Removing Y label because it's the same
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(x):,}')) # Y-axis formatting
plt.tight_layout()
plt.show()

The first chart clearly shows that cars with an automatic transmission (False) are significantly more expensive than those with a manual transmission (True). They have a noticeably higher median price and a much larger price spread, including the most expensive models in the dataset.
The second chart shows that cars offered by dealers (False) are statistically more expensive than those sold by private individuals (True). Prices at dealerships have a higher median and a wider range, while private sales are concentrated in the lower price bracket.

# Statistical Inference


In [ ]:
alpha = 0.05
test_set = cleaned_df['selling_price'].sample(frac=0.2, random_state=420321)  # 20% data sample for normality test
print(f"\nTest set shape: {len(test_set)} rows and 25 columns.")
print(f"\nH0: The distribution of Selling Price is normal.\nH1: The distribution of Selling Price is not normal.\nSignificance level (alpha): {alpha}")

# Shapiro-Wilk test for distribution normality

shapiro_test = scipy.stats.shapiro(test_set)
print("\n--- Shapiro-Wilk Test for normality of Selling Price distribution ---")
print(f"Test statistic: {shapiro_test.statistic}, p-value: {shapiro_test.pvalue}")

if shapiro_test.pvalue < alpha:
    print("We reject the null hypothesis: The distribution of Selling Price is not a normal distribution.")
else:
    print("We do not reject the null hypothesis: The distribution of Selling Price is a normal distribution.")

The test result yielded a p-value significantly smaller than 0.05. This means that we have strong grounds to reject the null hypothesis. This implies that the distribution of car prices in the examined dataset is not a normal distribution, which is also confirmed by previous observations from the histogram (strong positive skewness). The violation of the normality assumption can affect the interpretation of significance tests in classical linear regression, but for predictive purposes in machine learning, it is not a critical limitation.

# Correlation Matrix

In [ ]:
corr_matrix = cleaned_df.corr(method='pearson')
plt.figure(figsize=(18, 15)) # Size can be adjusted
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, annot_kws={"size": 8})
plt.title('Correlation Matrix for Car Dataset', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout() # Adjust layout so labels don't overlap
plt.show()

**Strongest POSITIVE correlations (the higher the value, the higher the price):**

    max_power: +0.75

This is the strongest positive indicator. It means that cars with more power (BHP) are definitely more expensive. This is very intuitive.

    brand_Other: +0.58

This is a very interesting correlation. The "other brand" variable groups rarer, more luxury brands that did not get their own column. Therefore, if a car belongs to this category, its price is significantly higher - likely related to the prestige phenomenon.

    torque: +0.50

Similar to power, higher torque is a characteristic of more expensive, more powerful cars.

    engine (engine capacity): +0.46

Cars with larger engine capacity (CC) tend to be more expensive. This is a moderate but still significant correlation.

**Strongest NEGATIVE correlations (the higher the value, the lower the price):**

    transmission_Manual: -0.59

This is the strongest negative correlation. It means that if a car has a manual transmission (transmission_Manual = 1), its price is clearly lower. This indicates that in this dataset, cars with automatic transmissions are more expensive.

    car_age: -0.41

A very logical relationship. The older the car (the higher the age), the lower its selling price.

    seller_type_Individual: -0.39

Prices of cars sold by private individuals are statistically lower than those offered by dealers.

    km_driven: -0.22

Higher mileage is associated with a lower price, but interestingly, this correlation is weaker than in the case of car age or transmission type.

# Simple and Multiple Linear Regression Model:


Simple linear regression - selling_price ~ max_power

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree

In [ ]:

Y = cleaned_df['selling_price']

simple_regression_features =['max_power'] 
simple_regression_features =[col for col in simple_regression_features if col in cleaned_df.columns]

for feature_name in simple_regression_features:
        print(f"\n--- Model: selling_price ~ {feature_name} ---")
        X_simple = cleaned_df[[feature_name]] # Must be a DataFrame (double square brackets)

        # Train-test split
        X_train_simple, X_test_simple, Y_train, Y_test = train_test_split(X_simple, Y, test_size=0.2, random_state=42)

        # Model building and training
        model_simple = LinearRegression()
        model_simple.fit(X_train_simple, Y_train)

        # Model coefficients
        print(f"Intercept (b0): {model_simple.intercept_:.2f}")
        print(f"Coefficient for {feature_name} (b1): {model_simple.coef_[0]:.2f}")
        print(f"Equation: selling_price = {model_simple.intercept_:.2f} + ({model_simple.coef_[0]:.2f} * {feature_name})")

        # Predictions
        Y_pred_train_simple = model_simple.predict(X_train_simple)
        Y_pred_test_simple = model_simple.predict(X_test_simple)

        # Model evaluation
        r2_train_simple = r2_score(Y_train, Y_pred_train_simple)
        r2_test_simple = r2_score(Y_test, Y_pred_test_simple)
        rmse_test_simple = np.sqrt(mean_squared_error(Y_test, Y_pred_test_simple))
        mae_test_simple = mean_absolute_error(Y_test, Y_pred_test_simple)

        print(f"R^2 (train): {r2_train_simple:.2f}")
        print(f"R^2 (test): {r2_test_simple:.2f}")
        print(f"RMSE (test): {rmse_test_simple:.2f}")
        print(f"MAE (test): {mae_test_simple:.2f}")



In [ ]:
# Visualization (optional, but can be useful)
plt.figure(figsize=(8, 6))
plt.scatter(X_test_simple, Y_test, color='blue', label='Actual data')
plt.plot(X_test_simple, Y_pred_test_simple, color='red', linewidth=2, label='Regression line')
plt.xlabel(feature_name)
plt.ylabel('Car Price (INR)')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(x):,}')) # Y-axis formatting
plt.title(f'Simple Linear Regression: selling_price vs {feature_name}')
plt.legend()
plt.show()

Multiple linear regression:


In [ ]:
multiple_regression_features = cleaned_df.drop(columns=['selling_price']).columns.tolist()

# we have a high correlation between 'engine','max_power' and 'torque', so we drop them to improve model stability

multiple_regression_features= cleaned_df.drop(columns=['selling_price', 'engine', 'torque'])

X_multiple_cols =[col for col in multiple_regression_features if col in cleaned_df.columns]


In [ ]:
X_multiple = cleaned_df[X_multiple_cols]
print("Selected features for the multiple model:", X_multiple_cols)

# Train-test split
X_train_multi, X_test_multi, Y_train_multi, Y_test_multi = train_test_split(X_multiple, Y, test_size=0.2, random_state=42)

# Model building and training
model_multi = LinearRegression()
model_multi.fit(X_train_multi, Y_train_multi)

# Model coefficients
print(f"\nIntercept (b0): {model_multi.intercept_:.2f}")
print("Coefficients (b1, b2, ...):")
for feature, coef in zip(X_multiple_cols, model_multi.coef_):
  print(f"  {feature}: {coef:.2f}")

# Predictions
Y_pred_train_multi = model_multi.predict(X_train_multi)
Y_pred_test_multi = model_multi.predict(X_test_multi)

# Model evaluation
r2_train_multi = r2_score(Y_train_multi, Y_pred_train_multi)
r2_test_multi = r2_score(Y_test_multi, Y_pred_test_multi)
rmse_test_multi = np.sqrt(mean_squared_error(Y_test_multi, Y_pred_test_multi))
mae_test_multi = mean_absolute_error(Y_test_multi, Y_pred_test_multi)

print(f"\nR^2 (train): {r2_train_multi:.2f}")
print(f"R^2 (test): {r2_test_multi:.2f}")
print(f"RMSE (test): {rmse_test_multi:.2f}")
print(f"MAE (test): {mae_test_multi:.2f}")

# Model errors

print("\nMetrics for the multiple regression model:")
print(f"  R^2 (test): {r2_test_multi:.2f}")
print(f"  RMSE (test): {rmse_test_multi:.2f}")
print(f"  MAE (test): {mae_test_multi:.2f}")


Interpretation of some results:
1. max_power: +11918.73 -> Every additional BHP raises the price by ~12k INR. Very strong positive impact.
2. transmission_Manual: -302529.27 -> Manual transmission lowers the predicted price by as much as ~302k INR compared to automatic. This is a key factor.
3. car_age: -35068.25 -> Every year of "age" lowers the price by ~35k INR.
4. seller_type_Individual: -180574.33 -> A car from a private individual is on average ~180k INR cheaper than from a dealer (baseline category).
5. brand_Other: +592743.39 -> Cars from rare, luxury brands are on average ~593k INR more expensive, which makes sense in the context of the current economic situation.

## Decision Trees instead of linear regression ##


In [ ]:

model_tree = DecisionTreeRegressor(max_depth=8, random_state=42)
model_tree.fit(X_train_multi, Y_train_multi)

Y_pred_train_tree = model_tree.predict(X_train_multi)
Y_pred_test_tree = model_tree.predict(X_test_multi)

r2_train_tree = r2_score(Y_train_multi, Y_pred_train_tree)
rmse_train_tree = np.sqrt(mean_squared_error(Y_train_multi, Y_pred_train_tree))
mae_train_tree = mean_absolute_error(Y_train_multi, Y_pred_train_tree)

r2_test_tree = r2_score(Y_test_multi, Y_pred_test_tree)
rmse_test_tree = np.sqrt(mean_squared_error(Y_test_multi, Y_pred_test_tree))
mae_test_tree = mean_absolute_error(Y_test_multi, Y_pred_test_tree)

print("--- Results for Decision Tree ---")

print(f"R^2 (train): {r2_train_tree:.2f}")
print(f"RMSE (train): {rmse_train_tree:.2f}")
print(f"MAE (train): {mae_train_tree:.2f}\n")

print(f"R^2 (test): {r2_test_tree:.2f}")
print(f"RMSE (test): {rmse_test_tree:.2f}")
print(f"MAE (test): {mae_test_tree:.2f}")


In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(x=Y_test_multi, y=Y_pred_test_tree, alpha=0.6)
# Adding y=x line
p1 = max(max(Y_pred_test_tree), max(Y_test_multi))
p2 = min(min(Y_pred_test_tree), min(Y_test_multi))
plt.plot([p1, p2], [p1, p2], 'r--')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(x):,}')) # Y-axis formatting
plt.title('Actual vs. Predicted Values (Decision Tree)')
plt.xlabel('Actual Prices - INR')
plt.ylabel('Predicted Prices - INR')
plt.show()

# Model evaluation and comparison:

In [ ]:
# Table comparing models
results_df = pd.DataFrame({
    'Model':['Simple Linear Regression', 'Multiple Linear Regression', 'Decision Tree'],
    'R^2 (test)':[r2_test_simple, r2_test_multi, r2_test_tree],
    'RMSE (test)':[rmse_test_simple, rmse_test_multi, rmse_test_tree],
    'MAE (test)':[mae_test_simple, mae_test_multi, mae_test_tree]
})

print("--- Comparison of model results ---")
print(results_df.round(2))

The simple regression model, based solely on engine power, achieved an R² of 0.59, meaning that power alone explains 59% of the price variance. This is a decent result for a single variable.
The multiple regression model significantly improved the results (R² ≈ 0.74), proving that incorporating additional features, such as car age or transmission type, allowed for a better fit. The RMSE and MAE errors also decreased.
The decision tree model turned out to be by far the best, achieving an R² of 0.96. This means that this model, thanks to its ability to capture non-linear relationships between features, explains as much as 96% of the price variance. Its prediction errors (RMSE and MAE) are also the lowest among all models.

# Conclusions  

The goal of the project was to investigate the factors affecting the price of used cars in India and to build a predictive model. This goal was fully achieved. The correlation analysis and model evaluation showed that the key price predictors are:
- Engine power (max_power) - the strongest positive impact.

- Car age (car_age) and transmission type (transmission_Manual) - the strongest factors lowering the price.

- Other significant factors include engine capacity, brand, and whether the seller is a private individual.

Among the tested models, the decision tree model showed the highest predictive effectiveness. Its ability to model complex, non-linear interactions between features allowed for a significant reduction in prediction errors compared to linear models. The analysis was based solely on ad data. The model does not take into account unquantifiable factors, such as the technical condition of the vehicle or service history. An exemplary next step to improve the results could be the application of more advanced models, such as Random Forest or Gradient Boosting, which often offer even higher accuracy and are more robust to overfitting.